<a href="https://colab.research.google.com/github/arthurmraesdsilva-lgtm/codespaces-express/blob/main/Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Design patterns para back-end**

Passo 1: Implementação monolítica

In [24]:
class EcommerceSystem:
    def __init__(self):
        # Inicializa o banco de dados temporário (em memória) do sistema
        self.users = []      # Armazena a lista de usuários cadastrados
        self.products = []   # Armazena a lista de produtos em estoque
        self.orders = []     # Armazena o histórico de pedidos feitos

    def add_user(self, user):
        # Adiciona um novo usuário ao sistema (Falta validação de duplicados)
        self.users.append(user)

    def add_product(self, product):
        # Adiciona um novo produto ao estoque (Falta validação de duplicados)
        self.products.append(product)

    def place_order(self, user, product):
        # Verifica se o produto solicitado existe na lista de produtos
        if product in self.products:
            # Se existir, vincula o usuário ao produto e salva na lista de pedidos
            self.orders.append((user, product))
            return "Order placed successfully"  # Encerra a função aqui em caso de sucesso

            # ERRO: Esta linha está desalinhada (deveria estar fora do 'if').
            # Como está depois de um 'return', este código nunca será executado (Código Morto).
            # Se o produto NÃO estiver na lista, a função retorna 'None' em vez desta mensagem.
            return "Product not available"

# --- Exemplo de uso do sistema ---

system = EcommerceSystem()         # Cria uma instância do sistema monolítico
system.add_user("Alice")           # Cadastra a usuária "Alice"
system.add_product("Laptop")       # Adiciona o produto "Laptop" ao estoque

# Executa o pedido e imprime o resultado na tela
print(system.place_order("Alice", "Laptop"))


Order placed successfully


Passo 2: Refatoração para microsserviços

In [23]:
class UserService:
    def __init__(self):
        # Banco de dados isolado: armazena apenas dados de usuários
        self.users = []

    def add_user(self, user):
        # Cadastra um novo usuário no serviço
        self.users.append(user)

    def check_user_exists(self, user):
        # Serviço expõe esta função para que outros serviços verifiquem a existência do usuário
        return user in self.users


class ProductService:
    def __init__(self):
        # Banco de dados isolado: armazena apenas dados de produtos/estoque
        self.products = []

    def add_product(self, product):
        # Adiciona um produto ao catálogo do serviço
        self.products.append(product)

    def check_availability(self, product):
        # Serviço expõe esta função para checar se o item está disponível para venda
        return product in self.products


class OrderService:
    def __init__(self):
        # Banco de dados isolado: armazena apenas o histórico de pedidos realizados
        self.orders = []

    # Injeção de Dependência: o método recebe as instâncias dos outros serviços para se comunicar com eles
    def place_order(self, user, product, product_service, user_service):

        # 1. Comunicação inter-serviços: Consulta o UserService para validar o cliente
        if not user_service.check_user_exists(user):
            return "Erro: Usuário não cadastrado"

        # 2. Comunicação inter-serviços: Consulta o ProductService para validar o estoque
        if not product_service.check_availability(product):
            return "Erro: Produto não disponível"

        # 3. Orquestração: Se ambas as validações passarem, o pedido é registrado localmente
        self.orders.append((user, product))
        return "Pedido realizado com sucesso!"


# --- Execução do Código (Simulação do Ecossistema) ---

# Instanciando os serviços de forma independente (cada um representa um microsserviço isolado)
user_service = UserService()
product_service = ProductService()
order_service = OrderService()

# Populando as bases de dados de cada serviço de maneira isolada
user_service.add_user("Alice")
product_service.add_product("Laptop")

# Teste 1: Fluxo ideal - Usuário e produto existem, pedido é criado com sucesso
print(order_service.place_order("Alice", "Laptop", product_service, user_service))

# Teste 2: Falha na validação de usuário - "Bob" não existe no UserService
print(order_service.place_order("Bob", "Laptop", product_service, user_service))

# Teste 3: Falha na validação de produto - "Smartphone" não existe no ProductService
print(order_service.place_order("Alice", "Smartphone", product_service, user_service))


Pedido realizado com sucesso!
Erro: Usuário não cadastrado
Erro: Produto não disponível


**Escalabilidade e performance em
arquitetura de aplicações back-end**

Passo 1: Implementação monolítica

In [20]:
class OnlineStore:
    def __init__(self):
        self.users = []
        self.products = []
        self.orders = []

    def register_user(self, user):
        self.users.append(user)

    def add_product(self, product):
        self.products.append(product)

    def place_order(self, user, product):
        if product in self.products:
            self.orders.append((user, product))
            return "Order placed successfully"
        return "Product not available"


# Exemplo de uso
store = OnlineStore()
store.register_user("Alice")
store.add_product("Laptop")
print(store.place_order("Alice", "Laptop"))

Order placed successfully


Passo 2: Refatoração para microsserviços

In [22]:
class UserService:
    def __init__(self):
        self.users = []

    def register_user(self, user):
        self.users.append(user)


class ProductService:
    def __init__(self):
        self.products = []

    def add_product(self, product):
        self.products.append(product)

    def check_availability(self, product):
        return product in self.products


class OrderService:
    def __init__(self):
        self.orders = []

    def place_order(self, user, product, product_service):
        if product_service.check_availability(product):
            self.orders.append((user, product))
            return "Order placed successfully"
        return "Product not available"


# Exemplo de uso com microsserviços
user_service = UserService()
product_service = ProductService()
order_service = OrderService()

user_service.register_user("Alice")
product_service.add_product("Laptop")

print(order_service.place_order("Alice", "Laptop", product_service))

Order placed successfully
